In [ ]:
# !pip install zenml
# !pip install pyngrok

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
NGROK_TOKEN = os.getenv("NGROK_TOKEN")
!ngrok authtoken {NGROK_TOKEN}

In [ ]:
import os, sys, shutil
from pathlib import Path

# full ZenML reset (local + global)
paths_to_remove = [
    Path(".zen"),  # project-local repo
    Path.home() / ".config" / "zenml",  # common global path
    Path(os.path.expandvars(r"%USERPROFILE%\.config\zenml")),
    Path(os.path.expandvars(r"%APPDATA%\zenml")),
]

for p in paths_to_remove:
    if p.exists():
        print("Removing:", p)
        shutil.rmtree(p, ignore_errors=True)

print("Kernel Python:", sys.executable)

# init from terminal to avoid errors

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import ClassifierMixin
from sklearn.svm import SVC
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

In [ ]:
# using zenml steps to define our pipeline blocks
from zenml import step
from typing_extensions import Annotated
from typing import Tuple


# Step 1: load the dataset and split it
@step
def importer() -> Tuple[
    Annotated[np.ndarray, "X_train"],
    Annotated[np.ndarray, "X_test"],
    Annotated[np.ndarray, "y_train"],
    Annotated[np.ndarray, "y_test"],
]:
    digits = load_digits()
    data = digits.images.reshape((len(digits.images), -1))
    
    X_train, X_test, y_train, y_test = train_test_split(
        data, digits.target, test_size=0.2, shuffle=False
    )
    
    return X_train, X_test, y_train, y_test

# Step 2: define the model and train it
@step
def svc_trainer(
    X_train: np.ndarray,
    y_train: np.ndarray
):
    model = SVC(gamma=0.001)
    model.fit(X_train, y_train)
    
    return model


# Step 3: eval model
@step
def eval(
    X_test: np.ndarray,
    y_test: np.ndarray,
    model: ClassifierMixin
):
    test_acc = model.score(X_test, y_test)
    print(f"Test accuracy = {test_acc}")
    
    return test_acc

In [ ]:
# Create pipline from steps
from zenml import pipeline

@pipeline
def digits_pipeline():
    # step 1
    X_train, X_test, y_train, y_test = importer() # -> use steps
    
    # step 2
    model = svc_trainer(X_train, y_train)
    
    # step 3
    eval(X_test, y_test, model)

In [ ]:
digits_svc_pipeline = digits_pipeline()
# digits_svc_pipeline.run(unlisted = True)